# Run experiment (Samara / Synthetic) + compare models

Thin example for the `pattern_recognition` experiment runner:

1. Import the package
2. Run two configs (EEGNet vs BaseCNN) — uses Samara if `Samara_data/` is present, otherwise `SyntheticBinary`
3. Load artifacts with `load_run`
4. Build a metrics table and compare runs
5. Colab install / Drive / run cells (optional section below)

Absolute imports only: `from pattern_recognition...`.


In [ ]:
from pathlib import Path

from pattern_recognition.experiment import run_experiment
from pattern_recognition.reporting import (
    compare_runs,
    load_run,
    metrics_table,
    plot_training_curves,
)

REPO_ROOT = Path(".").resolve()
# If this notebook is opened from notebooks/examples/, point at repo root:
if (REPO_ROOT / "configs").is_dir() is False and (REPO_ROOT.parent.parent / "configs").is_dir():
    REPO_ROOT = REPO_ROOT.parent.parent

print("repo root:", REPO_ROOT)


## Run two models (EEGNet vs BaseCNN)

Prefer checked-in Samara configs when data is available; otherwise run a short synthetic pair so the notebook stays usable offline.


In [ ]:
samara_path = REPO_ROOT / "Samara_data"
use_samara = samara_path.is_dir() and any(samara_path.glob("*.mat"))

if use_samara:
    print("Using Samara configs")
    config_eegnet = REPO_ROOT / "configs" / "samara_pz_eegnet_sc_n10.json"
    config_basecnn = REPO_ROOT / "configs" / "samara_pz_basecnn_sc_n10.json"
else:
    print("Samara_data missing — using SyntheticBinary EEGNet vs BaseCNN")
    common_data = {
        "pipeline": "SyntheticBinary",
        "params": {"n_train": 32, "n_val": 16, "n_channels": 1, "n_times": 64},
    }
    common_train = {
        "lr": 1e-3,
        "weight_decay": 0.0,
        "batch_size": 8,
        "num_epochs": 2,
        "step_size": 1,
        "gamma": 1.0,
        "save_model": True,
    }
    config_eegnet = {
        "name": "synthetic_eegnet",
        "seed": 0,
        "device": "cpu",
        "data": common_data,
        "model": {"name": "EEGNet", "params": {"n_channels": 1, "input_feat_dim": 64}},
        "train": common_train,
        "output_dir": str(REPO_ROOT / "results"),
    }
    config_basecnn = {
        "name": "synthetic_basecnn",
        "seed": 0,
        "device": "cpu",
        "data": common_data,
        "model": {"name": "BaseCNN", "params": {"n_channels": 1, "input_feat_dim": 64}},
        "train": common_train,
        "output_dir": str(REPO_ROOT / "results"),
    }

run_dir_eegnet = run_experiment(config_eegnet)
run_dir_basecnn = run_experiment(config_basecnn)
run_dirs = [run_dir_eegnet, run_dir_basecnn]
print("EEGNet run:", run_dir_eegnet)
print("BaseCNN run:", run_dir_basecnn)


## Load a single run and plot curves


In [ ]:
run = load_run(run_dir_eegnet)
print(run.metrics)
plot_training_curves(run)


## Metrics table + side-by-side comparison


In [ ]:
table = metrics_table(run_dirs)
try:
    display(table)
except NameError:
    print(table)
try:
    print(table.to_markdown(index=False))
except Exception:
    pass

compare_runs(run_dirs)


---

## Colab

Use these cells on Google Colab (GPU runtime recommended for Samara). Local runs can skip this section.


In [ ]:
# 1) Install from GitHub (or upload the repo and pip install -e .)
!pip install -q "git+https://github.com/Sidl419/pattern_recognition.git"

# Or, if the repo is already cloned / Drive-mounted:
# %cd /content/drive/MyDrive/pattern_recognition
# !pip install -q -e .


In [ ]:
# 2) Mount Drive if data / results live there
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 3) Run an experiment (inline config or load a JSON from Drive)
from pattern_recognition.experiment import run_experiment

config = {
    "name": "colab_samara_eegnet_sc",
    "seed": 42,
    "device": "cuda",  # or "auto" / "cpu"
    "data": {
        "pipeline": "SamaraWithinSubjectAverage",
        "params": {
            "path": "/content/drive/MyDrive/Samara_data/",
            "channel_idx": 1,
            "n_average": 10,
            "mode": "SC",
        },
    },
    "model": {
        "name": "EEGNet",
        "params": {"n_channels": 1, "input_feat_dim": 250},
    },
    "train": {
        "lr": 1e-4,
        "weight_decay": 1e-2,
        "batch_size": 64,
        "num_epochs": 50,
        "step_size": 20,
        "gamma": 0.5,
        "save_model": True,
    },
    "output_dir": "/content/drive/MyDrive/pattern_recognition_results/",
}

run_dir = run_experiment(config)


In [ ]:
# 4) Tables / plots from saved artifacts (single run)
from pattern_recognition.reporting import (
    load_run,
    plot_training_curves,
    metrics_table,
    compare_runs,
)

run = load_run(run_dir)
print(run.metrics)  # includes device_requested / device_resolved
plot_training_curves(run)


In [ ]:
# 5) Metrics comparison (after 2+ runs, e.g. EEGNet vs BaseCNN)
from pathlib import Path

run_dirs = sorted(Path("/content/drive/MyDrive/pattern_recognition_results/").glob("*"))
table = metrics_table(run_dirs)
display(table)  # Colab-friendly
compare_runs(run_dirs)
